# 00 · 获取并校验数据

本 Notebook 在 `smoke` 模式使用仓库内小样本，在 `full` 模式下载或读取 GitHub Release 数据包。所有文件通过 SHA256、schema 和行数校验后才写入活动数据合同。

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
import shutil
import subprocess
import urllib.request
import zipfile
from pathlib import Path

import pyarrow.parquet as pq


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'data' / 'data_manifest.json').exists():
            return candidate
    raise FileNotFoundError('repository root with data/data_manifest.json not found')


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def github_repo_slug(root: Path) -> str | None:
    result = subprocess.run(
        ['git', 'config', '--get', 'remote.origin.url'],
        cwd=root, capture_output=True, text=True, check=False,
    )
    remote = result.stdout.strip()
    if not remote:
        return None
    match = re.search(r'github\.com[/:]([^/]+)/([^/]+?)(?:\.git)?$', remote)
    if not match:
        return None
    owner, repository = match.groups()
    return f'{owner}/{repository}'


def github_release_url(root: Path, tag: str, asset_name: str) -> str | None:
    slug = github_repo_slug(root)
    return f'https://github.com/{slug}/releases/download/{tag}/{asset_name}' if slug else None


def github_release_asset_api_url(root: Path, tag: str, asset_name: str, token: str) -> str | None:
    slug = github_repo_slug(root)
    if not slug:
        return None
    request = urllib.request.Request(
        f'https://api.github.com/repos/{slug}/releases/tags/{tag}',
        headers={
            'Authorization': f'Bearer {token}',
            'Accept': 'application/vnd.github+json',
            'X-GitHub-Api-Version': '2022-11-28',
            'User-Agent': 'closing-auction-capacity-demo',
        },
    )
    with urllib.request.urlopen(request) as response:
        release = json.load(response)
    asset = next((item for item in release.get('assets', []) if item.get('name') == asset_name), None)
    return f'https://api.github.com/repos/{slug}/releases/assets/{asset["id"]}' if asset else None


def download_with_headers(url: str, destination: Path, token: str | None = None) -> None:
    headers = {'User-Agent': 'closing-auction-capacity-demo'}
    if token:
        headers.update({
            'Authorization': f'Bearer {token}',
            'Accept': 'application/octet-stream',
            'X-GitHub-Api-Version': '2022-11-28',
        })
    request = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(request) as source, destination.open('wb') as target:
        shutil.copyfileobj(source, target, length=1024 * 1024)


def validate_parquet(path: Path, contract: dict) -> None:
    parquet_file = pq.ParquetFile(path)
    if parquet_file.metadata.num_rows != int(contract['rows']):
        raise ValueError(f'row count mismatch for {path.name}')
    actual_columns = list(parquet_file.schema_arrow.names)
    if actual_columns != list(contract['column_names']):
        raise ValueError(f'schema mismatch for {path.name}')
    if sha256_file(path) != contract['sha256']:
        raise ValueError(f'SHA256 mismatch for {path.name}')
    forbidden = {'source_file', 'source_shard_id'} & set(actual_columns)
    if forbidden:
        raise ValueError(f'forbidden trace columns in {path.name}: {sorted(forbidden)}')


ROOT = find_repo_root()
RUN_MODE = os.environ.get('RUN_MODE', 'smoke').strip().lower()
if RUN_MODE not in {'smoke', 'full'}:
    raise ValueError('RUN_MODE must be smoke or full')
MANIFEST = json.loads((ROOT / 'data' / 'data_manifest.json').read_text(encoding='utf-8'))
OUTPUT_DIR = ROOT / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR = ROOT / 'data' / 'raw' / RUN_MODE
RAW_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
contracts = MANIFEST['smoke_files'] if RUN_MODE == 'smoke' else MANIFEST['files']
if RUN_MODE == 'smoke':
    source_dir = ROOT / 'data' / 'smoke'
    for contract in contracts.values():
        source = source_dir / contract['file_name']
        destination = RAW_DIR / contract['file_name']
        shutil.copy2(source, destination)
else:
    release_contract = MANIFEST['release_asset']
    zip_path = ROOT / 'release' / release_contract['file_name']
    if not zip_path.exists():
        token = os.environ.get('GH_TOKEN') or os.environ.get('GITHUB_TOKEN')
        url = os.environ.get('DATA_RELEASE_URL')
        if not url and token:
            url = github_release_asset_api_url(
                ROOT, MANIFEST['release_tag'], release_contract['file_name'], token
            )
        if not url:
            url = github_release_url(ROOT, MANIFEST['release_tag'], release_contract['file_name'])
        if not url:
            raise FileNotFoundError('Release ZIP absent and DATA_RELEASE_URL/origin unavailable')
        zip_path.parent.mkdir(parents=True, exist_ok=True)
        partial_path = zip_path.with_suffix(zip_path.suffix + '.part')
        try:
            download_with_headers(url, partial_path, token=token)
        except Exception as error:
            raise RuntimeError(
                'Release download failed. For a private repository, set GH_TOKEN or GITHUB_TOKEN '
                'with repository read access, or place the verified ZIP in release/.'
            ) from error
        if sha256_file(partial_path) != release_contract['sha256']:
            raise ValueError('downloaded Release .part SHA256 mismatch')
        partial_path.replace(zip_path)
    if sha256_file(zip_path) != release_contract['sha256']:
        raise ValueError('Release ZIP SHA256 mismatch')
    with zipfile.ZipFile(zip_path) as archive:
        expected_members = {contract['file_name'] for contract in contracts.values()}
        missing_members = expected_members - set(archive.namelist())
        if missing_members:
            raise ValueError(f'Release members missing: {sorted(missing_members)}')
        for member in sorted(expected_members):
            target = RAW_DIR / member
            with archive.open(member) as source, target.open('wb') as destination:
                shutil.copyfileobj(source, destination, length=1024 * 1024)

active_files = {}
for kind, contract in contracts.items():
    path = RAW_DIR / contract['file_name']
    validate_parquet(path, contract)
    active_files[kind] = path.relative_to(ROOT).as_posix()

active_contract = {
    'run_mode': RUN_MODE,
    'evaluation_scope': 'smoke_contract_only' if RUN_MODE == 'smoke' else MANIFEST['evaluation_scope'],
    'data_version': MANIFEST['data_version'],
    'cash_adv_grid': MANIFEST['cash_adv_grid'],
    'files': active_files,
}
active_path = OUTPUT_DIR / '00_active_data.json'
active_path.write_text(json.dumps(active_contract, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(active_contract, ensure_ascii=False, indent=2))
